# Production AI Systems

Build AI systems that scale — caching, rate limiting, cost control, fallback chains, and full observability.

**Topics:** LLM response caching, rate limiting, fallback chains, cost monitoring, observability with Langfuse, circuit breakers

## 1. LLM Response Caching with Redis

In [ ]:
import hashlib
import json
import os
from functools import wraps
from typing import Optional
import redis
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))
cache = redis.Redis(host='localhost', port=6379, decode_responses=True)

def cache_key(model: str, messages: list, temperature: float) -> str:
    """Deterministic cache key from request parameters."""
    payload = {'model': model, 'messages': messages, 'temperature': temperature}
    return f'llm_cache:{hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:32]}'

def cached_chat(
    messages: list,
    model: str = 'gpt-4o-mini',
    temperature: float = 0.0,  # only cache deterministic (temp=0) responses
    ttl_seconds: int = 86400,  # 24 hour TTL
) -> str:
    """Cache LLM responses — identical requests served from cache.
    
    Caching is only appropriate when temperature=0 (deterministic).
    Creative tasks should NOT be cached.
    """
    if temperature > 0:
        # Non-deterministic: skip cache
        response = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
        return response.choices[0].message.content
    
    key = cache_key(model, messages, temperature)
    
    if cached := cache.get(key):
        return json.loads(cached)['content']
    
    response = client.chat.completions.create(model=model, messages=messages, temperature=temperature)
    content = response.choices[0].message.content
    cache.setex(key, ttl_seconds, json.dumps({'content': content}))
    return content

# Cache savings analysis
cache_scenarios = [
    ('FAQ bot', '80%', '$0.10 → $0.02/day', 'Same questions repeated constantly'),
    ('Code reviewer', '30%', '$1.00 → $0.70/day', 'Same code reviewed multiple times'),
    ('Creative writer', '0%', 'No savings', 'Always temperature>0'),
    ('Data extractor', '60%', '$0.50 → $0.20/day', 'Same documents processed repeatedly'),
]
print('Cache effectiveness by use case:')
for use_case, hit_rate, savings, reason in cache_scenarios:
    print(f'  {use_case:<20}: cache hit rate={hit_rate:<6} savings={savings:<20} ({reason})')

## 2. Rate Limiting and Token Budget Management

In [ ]:
import time
from collections import deque
from threading import Lock
from dataclasses import dataclass

@dataclass
class RateLimits:
    requests_per_minute: int = 500
    tokens_per_minute: int = 150_000
    tokens_per_day: int = 2_000_000

class TokenBudgetManager:
    """Track and enforce token budgets per user/tenant."""
    
    def __init__(self, limits: RateLimits):
        self.limits = limits
        self._request_times: deque = deque()
        self._token_times: deque = deque()  # (timestamp, tokens)
        self._daily_tokens: int = 0
        self._daily_reset: float = time.time()
        self._lock = Lock()
    
    def _clean_old_requests(self):
        now = time.time()
        # Remove requests older than 1 minute
        while self._request_times and now - self._request_times[0] > 60:
            self._request_times.popleft()
        while self._token_times and now - self._token_times[0][0] > 60:
            self._token_times.popleft()
        # Reset daily counter
        if now - self._daily_reset > 86400:
            self._daily_tokens = 0
            self._daily_reset = now
    
    def check_and_record(self, estimated_tokens: int) -> tuple[bool, str]:
        with self._lock:
            self._clean_old_requests()
            now = time.time()
            
            if len(self._request_times) >= self.limits.requests_per_minute:
                return False, f'Rate limit: {self.limits.requests_per_minute} RPM exceeded'
            
            current_tokens = sum(t for _, t in self._token_times)
            if current_tokens + estimated_tokens > self.limits.tokens_per_minute:
                return False, f'Token rate limit: {self.limits.tokens_per_minute} TPM exceeded'
            
            if self._daily_tokens + estimated_tokens > self.limits.tokens_per_day:
                return False, f'Daily token budget of {self.limits.tokens_per_day:,} exceeded'
            
            self._request_times.append(now)
            self._token_times.append((now, estimated_tokens))
            self._daily_tokens += estimated_tokens
            return True, 'ok'

# Simulate rate limiting
manager = TokenBudgetManager(RateLimits(requests_per_minute=5, tokens_per_minute=1000, tokens_per_day=10000))
print('Rate limit simulation (5 RPM, 1000 TPM):')
for i in range(7):
    allowed, msg = manager.check_and_record(estimated_tokens=200)
    status = 'ALLOWED' if allowed else f'BLOCKED ({msg})'
    print(f'  Request {i+1}: {status}')

## 3. Fallback Chains — Primary → Cheaper Model

In [ ]:
from openai import OpenAI, RateLimitError, APIStatusError
import anthropic
import time
from typing import Callable

class FallbackChain:
    """Try models in order — fall back to cheaper/alternative on failure.
    
    Use cases:
    - Primary model rate limited → use backup
    - Primary model down → use competitor
    - High load → route some traffic to cheaper model
    """
    
    def __init__(self, providers: list[Callable]):
        self.providers = providers  # ordered list of callables
    
    def complete(self, prompt: str, **kwargs) -> tuple[str, str]:
        """Returns (response, provider_used)."""
        last_error = None
        for provider_fn in self.providers:
            try:
                result = provider_fn(prompt, **kwargs)
                return result, provider_fn.__name__
            except (RateLimitError, APIStatusError) as e:
                last_error = e
                print(f'Provider {provider_fn.__name__} failed: {e}. Trying next...')
                time.sleep(0.1)
            except Exception as e:
                last_error = e
                print(f'Unexpected error in {provider_fn.__name__}: {e}. Trying next...')
        raise RuntimeError(f'All providers failed. Last error: {last_error}')

oai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))
claude_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY', 'sk-ant-...'))

def gpt4o(prompt: str, max_tokens: int = 512) -> str:
    r = oai_client.chat.completions.create(model='gpt-4o', messages=[{'role':'user','content':prompt}], max_tokens=max_tokens)
    return r.choices[0].message.content

def gpt4o_mini(prompt: str, max_tokens: int = 512) -> str:
    r = oai_client.chat.completions.create(model='gpt-4o-mini', messages=[{'role':'user','content':prompt}], max_tokens=max_tokens)
    return r.choices[0].message.content

def claude_sonnet(prompt: str, max_tokens: int = 512) -> str:
    r = claude_client.messages.create(model='claude-sonnet-4-6', max_tokens=max_tokens, messages=[{'role':'user','content':prompt}])
    return r.content[0].text

chain = FallbackChain([gpt4o, claude_sonnet, gpt4o_mini])

print('Fallback chain: gpt4o → claude_sonnet → gpt4o_mini')
print()
print('Failure scenarios handled:')
scenarios = [
    ('gpt4o rate limited', 'Falls back to claude_sonnet'),
    ('OpenAI outage', 'Falls back to claude_sonnet (different provider)'),
    ('Both primary down', 'Falls back to gpt4o_mini (always-on, cheap)'),
]
for scenario, action in scenarios:
    print(f'  {scenario:<30}: {action}')

## 4. Cost Monitoring and Alerting

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
import json
from pathlib import Path

@dataclass
class LLMCallRecord:
    timestamp: str
    model: str
    prompt_tokens: int
    completion_tokens: int
    user_id: str
    feature: str  # which feature/endpoint triggered this call

# Cost per 1M tokens (input, output)
MODEL_COSTS = {
    'gpt-4o':           (2.50, 10.00),
    'gpt-4o-mini':      (0.15, 0.60),
    'claude-sonnet-4-6': (3.00, 15.00),
    'claude-haiku-4-5-20251001':  (0.25, 1.25),
}

def compute_cost(record: LLMCallRecord) -> float:
    if record.model not in MODEL_COSTS:
        return 0.0
    input_cost, output_cost = MODEL_COSTS[record.model]
    return (record.prompt_tokens * input_cost + record.completion_tokens * output_cost) / 1_000_000

def daily_cost_report(records: list[LLMCallRecord]) -> dict:
    by_model = {}
    by_feature = {}
    by_user = {}
    
    for r in records:
        cost = compute_cost(r)
        by_model[r.model] = by_model.get(r.model, 0) + cost
        by_feature[r.feature] = by_feature.get(r.feature, 0) + cost
        by_user[r.user_id] = by_user.get(r.user_id, 0) + cost
    
    total = sum(compute_cost(r) for r in records)
    return {'total': total, 'by_model': by_model, 'by_feature': by_feature, 'top_users': dict(sorted(by_user.items(), key=lambda x: -x[1])[:5])}

# Simulated usage data
import random
random.seed(42)
simulated_records = [
    LLMCallRecord('2025-01-15T10:30:00', random.choice(['gpt-4o-mini', 'gpt-4o', 'claude-sonnet-4-6']),
                  random.randint(100, 2000), random.randint(50, 500),
                  f'user_{random.randint(1,20):03d}', random.choice(['chat', 'rag', 'summary', 'extract']))
    for _ in range(200)
]

report = daily_cost_report(simulated_records)
print(f'Daily cost report ({len(simulated_records)} calls):')
print(f'  Total: ${report["total"]:.4f}')
print(f'\n  By model:')
for model, cost in sorted(report['by_model'].items(), key=lambda x: -x[1]):
    print(f'    {model:<30}: ${cost:.4f}')
print(f'\n  By feature:')
for feature, cost in sorted(report['by_feature'].items(), key=lambda x: -x[1]):
    print(f'    {feature:<20}: ${cost:.4f}')

## 5. Observability — Tracing with Langfuse

In [ ]:
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from openai import OpenAI

langfuse = Langfuse(
    public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
    secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
    host='https://cloud.langfuse.com',
)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'sk-...'))

@observe(name='rag-pipeline')  # auto-traces function + captures IO
def rag_pipeline(question: str, user_id: str) -> str:
    # Trace metadata
    langfuse_context.update_current_trace(user_id=user_id, tags=['rag', 'production'])
    
    # Step 1: Retrieve
    with langfuse_context.observe_span(name='retrieval', input={'query': question}) as span:
        retrieved_docs = [f'Doc about {question}']  # mock retrieval
        span.update(output={'n_docs': len(retrieved_docs)})
    
    # Step 2: Generate
    with langfuse_context.observe_generation(
        name='llm-generation',
        model='gpt-4o-mini',
        input={'question': question, 'context': retrieved_docs},
    ) as gen:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': f'Context: {retrieved_docs}\nQ: {question}'}],
        )
        answer = response.choices[0].message.content
        gen.update(output=answer, usage={'prompt_tokens': response.usage.prompt_tokens, 'completion_tokens': response.usage.completion_tokens})
    
    return answer

# What Langfuse captures automatically
captured_data = [
    ('Traces', 'Full pipeline execution with parent-child spans'),
    ('Generations', 'LLM calls with input/output/tokens/cost'),
    ('Latency', 'Per-span timing breakdown'),
    ('Errors', 'Exceptions with context'),
    ('User sessions', 'Group traces by user for session analysis'),
    ('Scores', 'Attach eval scores to traces for quality analysis'),
    ('Cost', 'Automatic cost calculation from token usage'),
]

print('Langfuse observability captures:')
for what, desc in captured_data:
    print(f'  {what:<20}: {desc}')
print()
print('Access traces at: https://cloud.langfuse.com')
print('Use to debug: which retrieval steps are slow? Which prompts cause hallucinations?')

## Exercises

1. **Semantic cache:** Extend the caching system to cache semantically similar queries (not just identical ones). Use embeddings to find cached responses within cosine similarity > 0.97. Measure cache hit rate improvement vs exact-match caching.

2. **Circuit breaker:** Implement a circuit breaker pattern for LLM calls: after 5 consecutive failures, open the circuit and immediately return a fallback response for 30 seconds, then try again. Track circuit state in Redis.

3. **Cost alert pipeline:** Build a system that: tracks running daily cost per user, sends a Slack webhook alert when any user exceeds $5/day, and automatically downgrades their model from gpt-4o to gpt-4o-mini for the rest of the day.